# Number Labyrinth — Generator Analysis

This notebook analyses and develops the number-generation algorithm for the
**Number Labyrinth** game (`src/games/NumberLabyrinth/index.jsx`).

## Scope (this notebook)
1. Python port of the **solution-path generator** (direct translation of the JS source).
2. A **board display** function that renders the grid and highlights the solution path.
3. A **BFS exhaustive enumerator** (`generate_solution_path_2`) that finds every valid path.

The full number-filling algorithm (steps 2–4) is left for a later cell once the
path-quality analysis is complete.

---
## 1 · Imports & level config

In [ ]:
import random
import math
from collections import deque
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Level config — mirrors LEVELS in index.jsx
LEVELS = {
    1: dict(cols=4, rows=5, min_val=0,  max_val=120),
    2: dict(cols=5, rows=5, min_val=0,  max_val=300),
    3: dict(cols=6, rows=6, min_val=0,  max_val=500),
    4: dict(cols=7, rows=7, min_val=0,  max_val=700),
    5: dict(cols=8, rows=8, min_val=8,  max_val=999),
}

---
## 2 · Solution-path generator (v1 — original JS port)

Direct Python translation of `generateSolutionPath` in `index.jsx`.

### Algorithm summary
* Target path length: between `0.35 × T` and `0.55 × T` cells (T = cols × rows).
* **Biased random walk** from `(0,0)` toward `(cols-1, rows-1)`:
  * At each step, only neighbours that can still reach the goal within the
    remaining budget are considered (Manhattan-distance pruning).
  * When the slack `(remaining steps − distance to goal) ≤ 3`, candidates are
    sorted by ascending distance so the walk commits toward the goal.
* Up to 500 attempts per target length; the outer loop tries increasing lengths
  until one succeeds.
* **Fallback**: a simple L-shaped path if all attempts fail.

In [ ]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]


def get_neighbors(c, r, cols, rows):
    """Return valid grid neighbours of (c, r)."""
    return [
        (c + dc, r + dr)
        for dc, dr in DIRS
        if 0 <= c + dc < cols and 0 <= r + dr < rows
    ]


def generate_solution_path(cols, rows, target_len, max_attempts=500):
    """
    Generate a simple (self-avoiding) path from (0,0) to (cols-1, rows-1)
    of exactly `target_len` cells.

    Returns the path as a list of (col, row) tuples, or None on failure.
    """
    gcol, grow = cols - 1, rows - 1

    def dist(c, r):
        return abs(gcol - c) + abs(grow - r)

    for _ in range(max_attempts):
        path = [(0, 0)]
        visited = {(0, 0)}
        c, r = 0, 0

        while len(path) < target_len:
            left = target_len - len(path)
            d = dist(c, r)

            if d == 0:
                break  # reached goal before filling target_len

            # Only neighbours reachable within remaining budget
            cands = [
                (nc, nr)
                for nc, nr in get_neighbors(c, r, cols, rows)
                if (nc, nr) not in visited and dist(nc, nr) <= left - 1
            ]

            if not cands:
                break

            random.shuffle(cands)

            # Bias toward goal when slack is tight
            if left - d <= 3:
                cands.sort(key=lambda p: dist(*p))

            c, r = cands[0]
            visited.add((c, r))
            path.append((c, r))

        if c == gcol and r == grow and len(path) == target_len:
            return path

    return None  # caller handles fallback


def get_solution_path(cols, rows):
    """
    Outer driver: tries increasing path lengths from min to max,
    returns the first successful path.  Falls back to L-shape.

    Mirrors the length-search loop in `getFieldNumbers` in index.jsx.
    """
    T = cols * rows
    min_len = math.floor(0.35 * T)
    max_len = math.floor(0.55 * T)

    for target_len in range(min_len, max_len + 1):
        path = generate_solution_path(cols, rows, target_len)
        if path is not None:
            return path

    # Fallback: simple L-shaped path along top row then right column
    path = [(0, 0)]
    pc, pr = 0, 0
    while pc < cols - 1:
        pc += 1
        path.append((pc, pr))
    while pr < rows - 1:
        pr += 1
        path.append((pc, pr))
    return path


# Quick smoke test
for lvl, cfg in LEVELS.items():
    p = get_solution_path(cfg['cols'], cfg['rows'])
    T = cfg['cols'] * cfg['rows']
    print(
        f"Level {lvl}  ({cfg['cols']}×{cfg['rows']})  "
        f"path len = {len(p)}  "
        f"(min {math.floor(0.35*T)} – max {math.floor(0.55*T)})  "
        f"ends at {p[-1]}"
    )

---
## 3 · Board display

`display_path(cols, rows, path)` draws the grid and overlays the solution path.

* Path cells are shaded in **light green**; non-path cells in **light beige**.
* The path is drawn as a sequence of arrows.
* Start cell `(0,0)` is marked **S**, goal cell `(cols-1, rows-1)` is marked **G**.
* Each cell shows its step index along the path (blank for non-path cells).

In [ ]:
def display_path(cols, rows, path, title=None, ax=None):
    """
    Render the grid and highlight `path` (list of (col, row) tuples).

    Parameters
    ----------
    cols, rows : int  — board dimensions
    path       : list of (col, row) — the solution path
    title      : optional string for the plot title
    ax         : optional existing Axes; a new figure is created if None
    """
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(cols * 0.85, rows * 0.85))

    path_set = set(path)
    path_index = {cell: i for i, cell in enumerate(path)}

    # ── Draw cells ──────────────────────────────────────────────────────────
    for r in range(rows):
        for c in range(cols):
            is_path = (c, r) in path_set
            is_light = (c + r) % 2 == 0

            if is_path:
                face = '#b6e8b6' if is_light else '#8fd98f'
            else:
                face = '#f5e6d0' if is_light else '#e8d5b7'

            rect = mpatches.FancyBboxPatch(
                (c, rows - 1 - r), 1, 1,
                boxstyle='square,pad=0',
                linewidth=0.8, edgecolor='#aaa', facecolor=face
            )
            ax.add_patch(rect)

            # Step index label
            if is_path:
                idx = path_index[(c, r)]
                label = 'S' if idx == 0 else ('G' if idx == len(path) - 1 else str(idx))
                ax.text(
                    c + 0.5, rows - 1 - r + 0.5, label,
                    ha='center', va='center',
                    fontsize=8, fontweight='bold', color='#1a5c1a'
                )

    # ── Draw path arrows ────────────────────────────────────────────────────
    for i in range(len(path) - 1):
        c1, r1 = path[i]
        c2, r2 = path[i + 1]
        # Convert to plot coords (y-axis flipped)
        x1, y1 = c1 + 0.5, rows - 1 - r1 + 0.5
        x2, y2 = c2 + 0.5, rows - 1 - r2 + 0.5
        ax.annotate(
            '', xy=(x2, y2), xytext=(x1, y1),
            arrowprops=dict(
                arrowstyle='->', color='#2a7a2a',
                lw=1.4,
                shrinkA=12, shrinkB=12
            )
        )

    # ── Axes cosmetics ───────────────────────────────────────────────────────
    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_aspect('equal')
    ax.axis('off')

    # Column and row labels
    for c in range(cols):
        ax.text(c + 0.5, rows + 0.15, str(c), ha='center', va='bottom',
                fontsize=7, color='#888')
    for r in range(rows):
        ax.text(-0.15, rows - 1 - r + 0.5, str(r), ha='right', va='center',
                fontsize=7, color='#888')

    if title:
        ax.set_title(title, fontsize=10, pad=4)

    if standalone:
        plt.tight_layout()
        plt.show()

In [ ]:
# ── Show one generated path per level ──────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 6))

for ax, (lvl, cfg) in zip(axes, LEVELS.items()):
    cols, rows = cfg['cols'], cfg['rows']
    path = get_solution_path(cols, rows)
    T = cols * rows
    display_path(
        cols, rows, path,
        title=f"Level {lvl}  ({cols}×{rows})\npath {len(path)}/{T} cells",
        ax=ax
    )

plt.suptitle('Number Labyrinth — solution paths by level', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 4 · Solution-path generator (v2 — BFS exhaustive enumeration)

### Validity constraint
A path is valid when every cell on it satisfies the **neighbour-count rule**:

| Cell role | Required path-neighbours (grid-adjacent cells also on the path) |
|---|---|
| Start `(0,0)` | exactly **1** |
| End `(cols-1, rows-1)` | exactly **1** |
| Any other path cell | exactly **2** |

### Key insight — incremental check
When extending the current path from cell `c` to a candidate `nc`, the
constraint reduces to a single fast check:

> **`nc` must have no already-visited spatial neighbour other than `c`.**

Why this is sufficient:
* `nc` currently has exactly 1 path-neighbour (`c`).  When we later leave `nc`
  it will gain its 2nd path-neighbour — satisfying the exactly-2 rule for
  intermediate cells.
* If `nc` already had a *second* visited neighbour `p ≠ c`, then once we step
  away from `nc` it would have 3 path-neighbours → violation.
* Adding `nc` cannot increase the path-neighbour count of any *already-visited*
  cell other than `c`, because the check above guarantees no such adjacency
  exists.
* For the goal cell the same check ensures it ends up with exactly 1
  path-neighbour (`c`).

### BFS approach
Because we need **all** valid paths (not just the shortest), the BFS queue
stores complete partial-path state: `(current_pos, frozenset(visited), path)`.
Each state is independent — no pruning based on previously-seen positions alone.
The non-touching constraint cuts the search space dramatically.

In [ ]:
def generate_solution_path_2(cols, rows):
    """
    BFS exhaustive enumeration of ALL valid paths from (0,0) to
    (cols-1, rows-1) satisfying the neighbour-count rule:
      - start and end cells have exactly 1 path-neighbour
      - every other path cell has exactly 2 path-neighbours
    (neighbours = grid-adjacent cells: up / down / left / right)

    Incremental validity check when extending path from c → nc:
      nc must have no already-visited spatial neighbour other than c.
    This single check is necessary and sufficient (see markdown above).

    BFS queue state: (current_pos, frozenset_visited, path_as_tuple)
    Complete path state is stored per entry because different partial
    paths reaching the same cell are NOT equivalent.

    Returns: list of all valid paths, each as a tuple of (col, row) pairs.
    """
    start = (0, 0)
    goal  = (cols - 1, rows - 1)

    def neighbors(c, r):
        return [
            (c + dc, r + dr)
            for dc, dr in DIRS
            if 0 <= c + dc < cols and 0 <= r + dr < rows
        ]

    # Queue entries: (current_pos, frozenset_of_visited, path_tuple)
    queue = deque([(start, frozenset([start]), (start,))])
    all_paths = []

    while queue:
        pos, visited, path = queue.popleft()

        for nc in neighbors(*pos):
            # Skip already-visited cells (self-avoiding walk)
            if nc in visited:
                continue

            # Core validity check: nc must touch the existing path ONLY at pos.
            # Any other visited spatial neighbour of nc would eventually give nc
            # (or that neighbour) more path-neighbours than allowed.
            if any(nb in visited and nb != pos for nb in neighbors(*nc)):
                continue

            new_visited = visited | {nc}
            new_path    = path + (nc,)

            if nc == goal:
                all_paths.append(new_path)
            else:
                queue.append((nc, new_visited, new_path))

    return all_paths


# ── Test on the two smallest boards ────────────────────────────────────────
for lvl in (1, 2):
    cfg = LEVELS[lvl]
    cols, rows = cfg['cols'], cfg['rows']
    paths = generate_solution_path_2(cols, rows)
    lengths = [len(p) for p in paths]
    print(
        f"Level {lvl}  ({cols}×{rows})  —  "
        f"{len(paths)} valid paths found   "
        f"path-length range: {min(lengths)}–{max(lengths)} cells"
    )